In [1]:
import os
import argparse


import sys

sys.path.append("../src")
sys.path.append("../config")

from utils import number_split, create_mix
from sampling_numbers import HateSpeech_DICT, SHAC_DICT

from pathlib import Path
import itertools
from tqdm.auto import tqdm
import numpy as np
import pandas as pd
import random
from sklearn import metrics
import pickle
from sklearn.metrics import precision_recall_fscore_support
import warnings

from process_HateSpeech import load_HateSpeech_dynGen, load_HateSpeech_wsf
from process_SHAC import load_process_SHAC

warnings.simplefilter("ignore")


# Load and Split

In [2]:
# df_dynGen = load_HateSpeech_dynGen()
# df_wsf = load_HateSpeech_wsf()

In [3]:
# df_dynGen = pd.read_csv("/bime-munin/xiruod/llama2_HateSpeech/n200/Inferences/inference_set-3636-quantization-epoch3-llama-2-7B-loraR-8-lambda1_1.0-lambda2_0.0-added_df_dynGen.csv")
# df_wsf = pd.read_csv("/bime-munin/xiruod/llama2_HateSpeech/n200/Inferences/inference_set-3636-quantization-epoch3-llama-2-7B-loraR-8-lambda1_1.0-lambda2_0.0-added_df_wsf.csv")


# df_dynGen = pd.read_csv("/bime-munin/xiruod/llama2_HateSpeech/n200/Inferences/inference_set-566-quantization-epoch3-llama-2-7B-loraR-8-lambda1_1.0-lambda2_0.0-added_df_dynGen.csv")
# df_wsf = pd.read_csv("/bime-munin/xiruod/llama2_HateSpeech/n200/Inferences/inference_set-566-quantization-epoch3-llama-2-7B-loraR-8-lambda1_1.0-lambda2_0.0-added_df_wsf.csv")

# df_dynGen = pd.read_csv("/bime-munin/xiruod/llama2_HateSpeech/n200/Inferences/inference_set-6621-quantization-epoch3-llama-2-7B-loraR-8-lambda1_1.0-lambda2_0.0-added_df_dynGen.csv")
# df_wsf = pd.read_csv("/bime-munin/xiruod/llama2_HateSpeech/n200/Inferences/inference_set-6621-quantization-epoch3-llama-2-7B-loraR-8-lambda1_1.0-lambda2_0.0-added_df_wsf.csv")


# df_dynGen = pd.read_csv("/bime-munin/xiruod/llama2_HateSpeech/n200/Inferences/inference_set-6621-quantization-epoch3-llama-2-7B-loraR-8-lambda1_1.5-lambda2_0.0-added_df_dynGen.csv")
# df_wsf = pd.read_csv("/bime-munin/xiruod/llama2_HateSpeech/n200/Inferences/inference_set-6621-quantization-epoch3-llama-2-7B-loraR-8-lambda1_1.5-lambda2_0.0-added_df_wsf.csv")



# df_dynGen = pd.read_csv("/bime-munin/xiruod/llama2_HateSpeech/n200/Inferences/inference_set-6621-quantization-epoch3-llama-2-7B-loraR-8-lambda1_1.5-lambda2_0.0-added_df_dynGen.csv")
# df_wsf = pd.read_csv("/bime-munin/xiruod/llama2_HateSpeech/n200/Inferences/inference_set-6621-quantization-epoch3-llama-2-7B-loraR-8-lambda1_1.5-lambda2_0.0-added_df_wsf.csv")



df_dynGen = pd.read_csv("/bime-munin/xiruod/llama2_HateSpeech/n200/Inferences/inference_set-566-quantization-epoch3-llama-2-7B-loraR-8-lambda1_1.5-lambda2_0.0-added_df_dynGen.csv")
df_wsf = pd.read_csv("/bime-munin/xiruod/llama2_HateSpeech/n200/Inferences/inference_set-566-quantization-epoch3-llama-2-7B-loraR-8-lambda1_1.5-lambda2_0.0-added_df_wsf.csv")



In [4]:
# df_shac = load_process_SHAC(replaceNA="all")

# df_shac_uw = df_shac.query("location == 'uw'").reset_index(drop=True)
# df_shac_mimic = df_shac.query("location == 'mimic'").reset_index(drop=True)


In [5]:
df_shac = load_process_SHAC(replaceNA="all")

df_shac_uw = df_shac.query("location == 'uw'").reset_index(drop=True)
df_shac_mimic = df_shac.query("location == 'mimic'").reset_index(drop=True)


In [6]:
n_test=200

p_pos_train_z0_ls = HateSpeech_DICT["Run-2"]["p_pos_train_z0_ls"]
p_pos_train_z1_ls = HateSpeech_DICT["Run-2"]["p_pos_train_z1_ls"]
p_mix_z1_ls = HateSpeech_DICT["Run-2"]["p_mix_z1_ls"]
    
z_Categories = ["dynGen","wsf"]
label = "label_binary"
split_label = "label_binary"
n_zCats = len(z_Categories)
txt_col = "text"
domain_col = "dfSource"
df0 = df_dynGen
df1 = df_wsf

In [7]:
c = HateSpeech_DICT[f"c_n200_566"]


############ Diff Out Dataset used in LoRA Fine-Tuning
dfs_used = create_mix(
    df0=df0,
    df1=df1,
    target=label,
    setting=c,
    sample=False,
    # seed=random.randint(0,1000),
    seed=222,
)

In [8]:
c

{'n_train': 800,
 'n_test': 200,
 'n_z0_pos_train': 40,
 'n_z0_neg_train': 360,
 'n_z0_pos_test': 30,
 'n_z0_neg_test': 70,
 'n_z1_pos_train': 200,
 'n_z1_neg_train': 200,
 'n_z1_pos_test': 30,
 'n_z1_neg_test': 70,
 'mix_param_dict': {'p_pos_train_z0': 0.1,
  'p_pos_train_z1': 0.5,
  'p_pos_train': 0.3,
  'p_pos_test': 0.3,
  'p_mix_z0': 0.5,
  'p_mix_z1': 0.5,
  'alpha_train': 5.0,
  'alpha_test': 1.0,
  'p_pos_test_z0': 0.3,
  'p_pos_test_z1': 0.3,
  'C_y': 0.3,
  'C_z': 0.5,
  'C_y_test': 0.3}}

# Prepare for Iterations

In [9]:
train_test_ratio = 4

df0 = df0[~df0[txt_col].isin(dfs_used["train"][txt_col])].reset_index(drop=True)
df0 = df0[~df0[txt_col].isin(dfs_used["test"][txt_col])].reset_index(drop=True)


df1 = df1[~df1[txt_col].isin(dfs_used["train"][txt_col])].reset_index(drop=True)
df1 = df1[~df1[txt_col].isin(dfs_used["test"][txt_col])].reset_index(drop=True)


############  Get Split Configs & Further Limit Sampling Set, if necessary

# numvals = 1023
# base = 1.1
# alpha_test_ls = np.power(base, np.arange(numvals)) / np.power(base, numvals // 2)
alpha_test_ls = np.float_power(10, np.linspace(start=-2, stop=2, num=40))


valid_full_settings = []
for combination in itertools.product(
    p_pos_train_z0_ls, p_pos_train_z1_ls, p_mix_z1_ls, alpha_test_ls
):

    number_setting = number_split(
        p_pos_train_z0=combination[0],
        p_pos_train_z1=combination[1],
        p_mix_z1=combination[2],
        alpha_test=combination[3],
        train_test_ratio=train_test_ratio,
        n_test=n_test,
        verbose=False,
    )

    if number_setting is not None:
        if np.all([number_setting[k] >= 10 for k in list(number_setting.keys())[:-1]]):
            valid_full_settings.append(number_setting)


valid_n_full_settings = []

for c in tqdm(valid_full_settings):
    c = c.copy()
    c["n_train"] = 0
    c["n_z0_pos_train"] = 1
    c["n_z0_neg_train"] = 1
    c["n_z1_pos_train"] = 1
    c["n_z1_neg_train"] = 1
    c["mix_param_dict"]["p_pos_train_z0"] = 0
    c["mix_param_dict"]["p_pos_train_z1"] = 0
    c["mix_param_dict"]["p_pos_train"] = 0
    c["mix_param_dict"]["alpha_train"] = 0

    # create train/test split according to stats
    dfs = create_mix(df0=df0, df1=df1, target=label, setting=c, sample=False, seed=222)

    if dfs is None:
        continue

    valid_n_full_settings.append(c)

tmp_df = [x["mix_param_dict"] for x in valid_n_full_settings]

tmp_df = pd.DataFrame(tmp_df)

tmp_df["alpha_train"] = tmp_df["alpha_train"].round(4)
tmp_df["C_y1"] = np.floor(tmp_df["C_y"] * 10) / 10
tmp_df["combination"] = valid_n_full_settings


  0%|          | 0/5240 [00:00<?, ?it/s]

# EM

In [10]:
cy_source = 0.3

In [11]:
# ## init
# pw1_0 = cy_source
# pt_w1 = cy_source

In [12]:
# pt_x_w1_ls = [0.2, 0.2, 0.2, 0.2, 0.2, 0.7, 0.6]


# pt_w0 = 1 - pt_w1
# pw1_s = pw1_0

# i_em = 0
# ct = 0
# while 1:
#     pw1_s_pre = pw1_s
#     pw0_s = 1 - pw1_s
    
#     pw1_x_s_ls = []
    
#     for pt_x_w1 in pt_x_w1_ls:
#         pt_x_w0 = 1 - pt_x_w1

#         deno = pw0_s/pt_w0 * pt_x_w0 + pw1_s/pt_w1 * pt_x_w1
#         pw1_x_s = pw1_s/pt_w1 * pt_x_w1 / deno
#         pw1_x_s_ls.append(pw1_x_s)
        
    
#     pw1_s = np.mean(pw1_x_s_ls)
#     i_em += 1
#     print(pw1_s_pre)
    
#     if abs(pw1_s_pre - pw1_s) < 0.0001:
#         ct += 1
#         if ct == 5:
#             break
    

In [13]:
# pw1_s

# i_em

# pw1_x_s_ls

In [14]:
def emUpdate(pt_x_w1_ls, pw1_0, pt_w1):

    pt_w0 = 1 - pt_w1
    pw1_s = pw1_0

    i_em = 0
    ct = 0
    while 1:
        pw1_s_pre = pw1_s
        pw0_s = 1 - pw1_s
        
        pw1_x_s_ls = []

        for pt_x_w1 in pt_x_w1_ls:
            pt_x_w0 = 1 - pt_x_w1
            
            deno = pw0_s/pt_w0 * pt_x_w0 + pw1_s/pt_w1 * pt_x_w1
            pw1_x_s = pw1_s/pt_w1 * pt_x_w1 / deno
            pw1_x_s_ls.append(pw1_x_s)


        pw1_s = np.mean(pw1_x_s_ls)
        i_em += 1

        if abs(pw1_s_pre - pw1_s) < 0.0001:
            ct += 1
            if ct == 5:
                break

    return pw1_x_s_ls, pw1_s

In [15]:
pt_x_w1_ls = [0.2, 0.2, 0.2, 0.2, 0.2, 0.7, 0.6]

emUpdate(pt_x_w1_ls=pt_x_w1_ls, pw1_0=cy_source, pt_w1=cy_source)

([0.3402493880214517,
  0.3402493880214517,
  0.3402493880214517,
  0.3402493880214517,
  0.3402493880214517,
  0.8279844025914581,
  0.7557607184360747],
 0.46928458016211305)

# Eval

In [ ]:
runs = 5


random.seed(123)
record_valid_settings_n = []
auprc_weightsEdited = []
auprc_weightsEdited_df0 = []
auprc_weightsEdited_df1 = []
auprc_weightsEdited_emUpdated = []
auprc_weightsEdited_emUpdated_df0 = []
auprc_weightsEdited_emUpdated_df1 = []
f1_weightsEdited = []
f1_weightsEdited_df0 = []
f1_weightsEdited_df1 = []
f1_emEdited = []
f1_emEdited_df0 = []
f1_emEdited_df1 = []

for iRun in range(runs):
    _rand = random.randint(0, 2**32 - 1)
    _n_setting = 0

    print(_rand)

    print(iRun)
    for c in tqdm(valid_full_settings, 
                  # file=open(log_f, "w")
                 ):

        c = c.copy()

        dfs = create_mix(
            df0=df0,
            df1=df1,
            target=label,
            setting=c,
            sample=False,
            seed=_rand,
        )

        if dfs is None:
            continue

        # ##### NTOE: for shorter version!!!
        # if args.sampleValidSettings:
        #     if round(c['mix_param_dict']['alpha_train'], 4) not in [1, 2, 0.5, 4, 0.25, 6, 0.1667]:
        #         continue

        # _n_setting += 1
        # if _n_setting % args.percent != 0:
        #     continue

        c["run"] = iRun
        record_valid_settings_n.append(c)

        y_train = dfs["train"][label]
        y_test = dfs["test"][label]

        n_test = len(y_test)

        y_probs_auprc_weightsEdited = dfs["test"][["ycat_0", "ycat_1"]].values
        
        tmp = emUpdate(pt_x_w1_ls=dfs["test"]["ycat_1"].values, pw1_0=cy_source, pt_w1=cy_source)
        y_probs_auprc_emUpdated = np.vstack([1-np.array(tmp[0]), np.array(tmp[0])]).T
        
        
        ret = c

        ret_code = 1

        auprc_weightsEdited.append(
            metrics.average_precision_score(
                y_true=y_test, y_score=y_probs_auprc_weightsEdited[:, 1]
            )
        )
        auprc_weightsEdited_df0.append(
            metrics.average_precision_score(
                y_true=y_test[dfs["test"][domain_col] == z_Categories[0]],
                y_score=y_probs_auprc_weightsEdited[
                    dfs["test"][domain_col] == z_Categories[0], 1
                ],
            )
        )
        auprc_weightsEdited_df1.append(
            metrics.average_precision_score(
                y_true=y_test[dfs["test"][domain_col] == z_Categories[1]],
                y_score=y_probs_auprc_weightsEdited[
                    dfs["test"][domain_col] == z_Categories[1], 1
                ],
            )
        )
        
        
        auprc_weightsEdited_emUpdated.append(
            metrics.average_precision_score(
                y_true=y_test, y_score=y_probs_auprc_emUpdated[:, 1]
            )
        )
        auprc_weightsEdited_emUpdated_df0.append(
            metrics.average_precision_score(
                y_true=y_test[dfs["test"][domain_col] == z_Categories[0]],
                y_score=y_probs_auprc_emUpdated[
                    dfs["test"][domain_col] == z_Categories[0], 1
                ],
            )
        )
        auprc_weightsEdited_emUpdated_df1.append(
            metrics.average_precision_score(
                y_true=y_test[dfs["test"][domain_col] == z_Categories[1]],
                y_score=y_probs_auprc_emUpdated[
                    dfs["test"][domain_col] == z_Categories[1], 1
                ],
            )
        )
        
        t = precision_recall_fscore_support(
            y_true=y_test,
            y_pred=y_probs_auprc_weightsEdited[:, 1] > 0.5,
            average="binary",
            pos_label=1,
        )
        t_df0 = precision_recall_fscore_support(
            y_true=y_test[dfs["test"][domain_col] == z_Categories[0]],
            y_pred=y_probs_auprc_weightsEdited[
                dfs["test"][domain_col] == z_Categories[0], 1
            ]
            > 0.5,
            average="binary",
            pos_label=1,
        )
        t_df1 = precision_recall_fscore_support(
            y_true=y_test[dfs["test"][domain_col] == z_Categories[1]],
            y_pred=y_probs_auprc_weightsEdited[
                dfs["test"][domain_col] == z_Categories[1], 1
            ]
            > 0.5,
            average="binary",
            pos_label=1,
        )
        
        
        t_em = precision_recall_fscore_support(
            y_true=y_test,
            y_pred=y_probs_auprc_emUpdated[:, 1] > 0.5,
            average="binary",
            pos_label=1,
        )
        t_em_df0 = precision_recall_fscore_support(
            y_true=y_test[dfs["test"][domain_col] == z_Categories[0]],
            y_pred=y_probs_auprc_emUpdated[
                dfs["test"][domain_col] == z_Categories[0], 1
            ]
            > 0.5,
            average="binary",
            pos_label=1,
        )
        t_em_df1 = precision_recall_fscore_support(
            y_true=y_test[dfs["test"][domain_col] == z_Categories[1]],
            y_pred=y_probs_auprc_emUpdated[
                dfs["test"][domain_col] == z_Categories[1], 1
            ]
            > 0.5,
            average="binary",
            pos_label=1,
        )
        
        # precision_weightsEdited.append(t[0])
        # recall_weightsEdited.append(t[1])
        f1_weightsEdited.append(t[2])
        f1_emEdited.append(t_em[2])
        # precision_weightsEdited_df0.append(t_df0[0])
        # recall_weightsEdited_df0.append(t_df0[1])
        f1_weightsEdited_df0.append(t_df0[2])
        f1_emEdited_df0.append(t_em_df0[2])
        # precision_weightsEdited_df1.append(t_df1[0])
        # recall_weightsEdited_df1.append(t_df1[1])
        f1_weightsEdited_df1.append(t_df1[2])
        f1_emEdited_df1.append(t_em_df1[2])


############  Put Results in DataFrame, with extra information (a little redundant)

# organize results in DataFrame
df_eval = pd.DataFrame(
    {
        "auprc_weightsEdited": auprc_weightsEdited,
        "auprc_weightsEdited_df0": auprc_weightsEdited_df0,
        "auprc_weightsEdited_df1": auprc_weightsEdited_df1,
        "auprc_weightsEdited_emUpdated": auprc_weightsEdited_emUpdated,
        "auprc_weightsEdited_emUpdated_df0": auprc_weightsEdited_emUpdated_df0,
        "auprc_weightsEdited_emUpdated_df1": auprc_weightsEdited_emUpdated_df1,
        # "precision_weightsEdited": precision_weightsEdited,
        # "recall_weightsEdited": recall_weightsEdited,
        "f1_weightsEdited": f1_weightsEdited,
        "f1_emEdited": f1_emEdited,
        # "precision_weightsEdited_df0": precision_weightsEdited_df0,
        # "recall_weightsEdited_df0": recall_weightsEdited_df0,
        "f1_weightsEdited_df0": f1_weightsEdited_df0,
        "f1_emEdited_df0": f1_emEdited_df0,
        # "precision_weightsEdited_df1": precision_weightsEdited_df1,
        # "recall_weightsEdited_df1": recall_weightsEdited_df1,
        "f1_weightsEdited_df1": f1_weightsEdited_df1,
        "f1_emEdited_df1": f1_emEdited_df1,
        # "auprc_logistic_vanilla_df0": auprc_logistic_vanilla_df0,
        # "auprc_logistic_vanilla_df1": auprc_logistic_vanilla_df1,
        # "precision_vanilla":precision_vanilla,
        # "recall_vanilla":recall_vanilla,
        # "f1_vanilla":f1_vanilla,
        # "precision_vanilla_df0":precision_vanilla_df0,
        # "recall_vanilla_df0":recall_vanilla_df0,
        # "f1_vanilla_df0":f1_vanilla_df0,
        # "precision_vanilla_df1":precision_vanilla_df1,
        # "recall_vanilla_df1":recall_vanilla_df1,
        # "f1_vanilla_df1":f1_vanilla_df1,
    }
)


for k in record_valid_settings_n[0]["mix_param_dict"].keys():
    df_eval[k] = [_dict["mix_param_dict"][k] for _dict in record_valid_settings_n]

for k in record_valid_settings_n[0].keys():
    if k != "mix_param_dict":
        df_eval[k] = [_dict[k] for _dict in record_valid_settings_n]


# outname = f"{outdir}/{name_general}.pkl"
# with open(outname, "wb") as f:
#     pickle.dump(df_eval, file=f)


224899942
0


  0%|          | 0/5240 [00:00<?, ?it/s]

1749090055
1


  0%|          | 0/5240 [00:00<?, ?it/s]

163868757
2


  0%|          | 0/5240 [00:00<?, ?it/s]

In [ ]:
y_probs_auprc_weightsEdited[:10]

In [ ]:
y_probs_auprc_emUpdated[:10]

# Plot

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
sns.set_theme()
sns.set_context("paper")

In [ ]:
_df_plt = df_eval[df_eval['C_y'].isin([0.3,0.5,0.7])].copy()


fig, ax = plt.subplots(1,2, figsize=(15,7), sharey=True)

sns.lineplot(data=_df_plt, x="alpha_test", y="auprc_weightsEdited", hue='C_y', ax= ax[0])
sns.lineplot(data=_df_plt, x="alpha_test", y="auprc_weightsEdited_emUpdated", hue='C_y', ax= ax[1])


for i in range(2):
    ax[i].set_xscale('log')
    xlab = r"$\alpha_{test}$"
    ax[i].set_xlabel(xlab)


In [ ]:
_df_plt = df_eval[df_eval['C_y'].isin([0.3,0.5,0.7])].copy()


fig, ax = plt.subplots(1,2, figsize=(15,7), sharey=True)

sns.lineplot(data=_df_plt, x="alpha_test", y="f1_weightsEdited", hue='C_y', ax= ax[0])
sns.lineplot(data=_df_plt, x="alpha_test", y="f1_emEdited", hue='C_y', ax= ax[1])


for i in range(2):
    ax[i].set_xscale('log')
    xlab = r"$\alpha_{test}$"
    ax[i].set_xlabel(xlab)


In [ ]:
_df_plt = df_eval[df_eval['C_y'].isin([0.3,0.5,0.7])].copy()


fig, ax = plt.subplots(1,2, figsize=(15,7), sharey=True)

sns.lineplot(data=_df_plt, x="alpha_test", y="f1_weightsEdited_df0", hue='C_y', ax= ax[0])
sns.lineplot(data=_df_plt, x="alpha_test", y="f1_emEdited_df0", hue='C_y', ax= ax[1])


for i in range(2):
    ax[i].set_xscale('log')
    xlab = r"$\alpha_{test}$"
    ax[i].set_xlabel(xlab)


In [ ]:
_df_plt = df_eval[df_eval['C_y'].isin([0.3,0.5,0.7])].copy()


fig, ax = plt.subplots(1,2, figsize=(15,7), sharey=True)

sns.lineplot(data=_df_plt, x="alpha_test", y="f1_weightsEdited_df1", hue='C_y', ax= ax[0])
sns.lineplot(data=_df_plt, x="alpha_test", y="f1_emEdited_df1", hue='C_y', ax= ax[1])


for i in range(2):
    ax[i].set_xscale('log')
    xlab = r"$\alpha_{test}$"
    ax[i].set_xlabel(xlab)


In [ ]:
df_eval.groupby("C_y", as_index=False).size().sort_values('size', ascending=False)

In [ ]:
# _df_plt = df_eval[df_eval['C_y'].isin([0.6])].copy()
_df_plt = df_eval.copy()


fig, ax = plt.subplots(1,1, figsize=(7,5), sharey=True, )

sns.lineplot(data=_df_plt, x="alpha_test", y="f1_weightsEdited", c='orange', label="Unadjusted", ax=ax)


sns.lineplot(data=_df_plt, x="alpha_test", y="f1_emEdited", label="Adjusted", c='blue', ax=ax)
ax.legend()
# ax.set_xlim(0.5, 10)
ax.set_xscale('log')
xlab = r"$\alpha_{test}$"
ax.set_xlabel(xlab)
